In [ ]:
from   ray.tune.schedulers import ASHAScheduler
import ray.cloudpickle as pickle
from   ray import tune
from   ray import train
# from ray.train import Checkpoint, get_checkpoint

In [4]:
from train import GenerateModel, federate_model
from torch.utils.data import DataLoader, TensorDataset
from metrics import eval_model
import torch
import os

In [ ]:
working_dir = os.path.split(os.getcwd())[0]
X_test = torch.load(os.path.join(working_dir,"Data","X_test.pt"))
Y_test = torch.load(os.path.join(working_dir,"Data","Y_test.pt"))

model = GenerateModel(table_path     = os.path.join(working_dir,"Model","processed_full.w2v"),
                      num_of_filters = 15,
                      kernel_size    = 5)

def load_data():
    working_dir = os.path.split(os.getcwd())[0]
    X_val  = torch.load(os.path.join(working_dir,"Data","X_val.pt"))
    Y_val  = torch.load(os.path.join(working_dir,"Data","Y_val.pt"))
    return DataLoader(TensorDataset(X_val,Y_val),batch_size=32,shuffle=False)

In [14]:
val_loader = load_data()

In [6]:
def train_model(config):
    model = GenerateModel(table_path = os.path.join(working_dir,"Model","processed_full.w2v"),
                      num_of_filters = 15,
                      kernel_size    = 5)
    ########################################
    eval_model(model    = model,
               device      = torch.device("cpu"),
               data_loader = load_data() # val_loader
               )
    return 
    ########################################
    checkpoint_data = {
        "epoch": epoch,
        "net_state_dict": net.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
    }
    with tempfile.TemporaryDirectory() as checkpoint_dir:
        data_path = Path(checkpoint_dir) / "data.pkl"
        with open(data_path, "wb") as fp:
            pickle.dump(checkpoint_data, fp)

        checkpoint = tune.Checkpoint.from_directory(checkpoint_dir)
        tune.report(
            {"loss": val_loss / val_steps},
            checkpoint=checkpoint,
        )

    print("Finished Training")

In [7]:
train_model(config = None)

TypeError: 'int' object is not iterable

In [ ]:
num_samples    = 5; # trials
max_num_epochs = 200; # Max epochs
gpus_per_trial = 1
data_dir = os.path.abspath("./data")
load_data(data_dir)
config = {
    "l1": tune.choice([10,15,20]), # Num of Filters
    "l2": tune.choice([3,4,5]),    # Filtern size
    "lr": tune.loguniform(0.0001, 0.1),
    "batch_size": tune.choice([8, 16, 32, 64]),
}
scheduler = ASHAScheduler(
    metric="loss",
    mode="min",
    max_t=max_num_epochs,
    grace_period=1,
    reduction_factor=2,
)
X = np.load('X_train.npy')
total_samples = X.shape[0]
data_points = list(random.choices(population=[0,1,2],k = total_samples))
result = tune.run(
    partial(train_model, data_dir=data_dir,data_points = data_points),
    resources_per_trial={"cpu": 2, "gpu": gpus_per_trial},
    config=config,
    num_samples=num_samples,
    scheduler=scheduler,
)